# Mission 3: Selfie Compressor & Eigenfaces 📸

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/03_selfie_compressor.ipynb)

> **From your syllabus**
> - *Linear Algebra*: the singular value decomposition
> - *ML-II*, Unit IV: PCA (maximum variance, dimensionality reduction)

**Your mission:** shrink a photo to a fraction of its size using only the SVD, and see exactly how much you lose.

> 📊 **Progress tracking: please read.** The next cell draws your progress map and records which exercises you finish. It sends your **name, roll number, exercise ID and time** to the instructor's progress sheet for this workshop, nothing else (no files, nothing from your Google account). Its full code is in the cell, so you can read it. Prefer not to be tracked? Type `anonymous` as your name, or set `TRACKING = False`; the map still works for you.
>
> Colab may ask *"Do you trust this code?"* because this notebook comes from GitHub (`anshid/MathMeetsML`, open source, MIT licence). That is expected: choose **Run anyway**.

In [ ]:
#@title Progress map (run me first; the code is short, read it)
# Records which exercises you finish so you can see your progress map and the instructor can see how the class is doing.
# Sent: your name, roll number, exercise id and time. Nothing else. Type "anonymous" as your name (or set TRACKING = False) to send nothing.
TRACKING = True
MISSION = 3

import json, os, time, base64, urllib.parse, urllib.request
from IPython.display import HTML, display, update_display

CLOUD_URL = "https://script.google.com/macros/s/AKfycbxekbOkIE5oRVA319mhEENS1phVLt7gPfRUernniP8kMopgIqUNHNN37xjF_3XZEuQWwQ/exec"
TOKEN = "3F5y-xKcPbZZPT49"
LAN_URL = os.environ.get("MML_LAN", "")
MISSIONS = {"0": {"title": "Ignite", "checkpoints": ["m00-ready", "m00-done"]}, "1": {"title": "Golf", "checkpoints": ["m01-grad", "m01-done"]}, "2": {"title": "Digit Reader", "checkpoints": ["m02-grad", "m02-done"]}, "3": {"title": "Selfie", "checkpoints": ["m03-ey", "m03-pca", "m03-done"]}, "4": {"title": "Mini-Google", "checkpoints": ["m04-markov", "m04-done"]}, "5": {"title": "Tiny AI", "checkpoints": ["m05-backprop", "m05-done"]}}
STORE = "progress.json"
SINKS = [u for u in (CLOUD_URL, LAN_URL) if u]


def _load():
    try:
        with open(STORE) as f:
            return json.load(f)
    except Exception:
        return {"name": "", "roll": "", "done": {}, "queue": []}


def _save(s):
    try:
        with open(STORE, "w") as f:
            json.dump(s, f)
    except Exception:
        pass


def _who(s):
    if s["roll"]:
        return
    if os.environ.get("MML_TRACKING") == "off":
        s["name"], s["roll"] = "anonymous", "-"
        return
    try:
        s["name"] = input("Your name (or 'anonymous'): ").strip() or "anonymous"
        s["roll"] = input("Roll number: ").strip() or "-"
    except Exception:
        s["name"], s["roll"] = "anonymous", "-"


def _sending(s):
    return TRACKING and s["name"].lower() != "anonymous" and os.environ.get("MML_TRACKING") != "off"


def _post(url, ev):
    body = json.dumps(dict(ev, token=TOKEN)).encode()
    urllib.request.urlopen(urllib.request.Request(url, body, {"Content-Type": "text/plain"}), timeout=2).read()


def sync():
    """Send queued exercises to the instructor. Silent if offline; they stay queued."""
    s = _load()
    if _sending(s):
        dead = set()
        for ev in s["queue"]:
            for u in SINKS:
                if u in dead or u in ev["sent"]:
                    continue
                try:
                    _post(u, ev)
                    ev["sent"].append(u)
                except Exception:
                    dead.add(u)
        s["queue"] = [e for e in s["queue"] if any(u not in e["sent"] for u in SINKS)]
    _save(s)
    return s


def progress_code():
    """Offline? Paste this code to the instructor (chat or USB) instead."""
    s = _load()
    print(base64.b64encode(json.dumps({"name": s["name"], "roll": s["roll"], "done": s["done"]}).encode()).decode())


def _status(s, n):
    ids = MISSIONS[str(n)]["checkpoints"]
    got = sum(1 for c in ids if c in s["done"])
    if got == len(ids):
        return "done", got, len(ids)
    if n == MISSION or got:
        return "now", got, len(ids)
    return "todo", got, len(ids)


def _html(s):
    cols = {"done": "#2e9e5b", "now": "#3b82f6", "todo": "#8a8f98"}
    parts, stats = [], [_status(s, n) for n in range(len(MISSIONS))]
    for n, (st, got, tot) in enumerate(stats):
        x = 45 + n * 108
        title = MISSIONS[str(n)]["title"]
        if n:
            ok = stats[n - 1][0] == "done"
            parts.append(f'<line x1="{x - 108 + 20}" y1="40" x2="{x - 20}" y2="40" stroke="{cols["done"] if ok else cols["todo"]}" stroke-width="3" stroke-dasharray="{"0" if ok else "5 4"}"/>')
        if st == "done":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="{cols[st]}"/><path d="M{x - 8} 40 l6 7 l11 -14" stroke="#fff" stroke-width="3.5" fill="none" stroke-linecap="round" stroke-linejoin="round"/>')
        elif st == "now":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="4"><animate attributeName="r" values="18;22;18" dur="1.8s" repeatCount="indefinite"/></circle><text x="{x}" y="45" text-anchor="middle" font-size="14" font-weight="700" fill="{cols[st]}">{got}/{tot}</text>')
        else:
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="2.5" stroke-dasharray="4 3"/><text x="{x}" y="45" text-anchor="middle" font-size="14" fill="{cols[st]}">{n}</text>')
        parts.append(f'<text x="{x}" y="82" text-anchor="middle" font-size="12" fill="currentColor" opacity="{1 if st != "todo" else 0.55}">M{n} {title}</text>')
    finished = [n for n, t in enumerate(stats) if t[0] == "done"]
    todo = [f"M{n}" for n, t in enumerate(stats) if t[0] == "todo"]
    line = f"<b>{len(finished)} of {len(stats)} missions finished</b> &middot; now in <b>Mission {MISSION}</b> ({stats[MISSION][1]}/{stats[MISSION][2]} exercises)"
    line += f" &middot; not visited yet: {', '.join(todo)}" if todo else " &middot; every mission visited"
    who = f"{s['name']} ({s['roll']})" if s["roll"] else ""
    return (f'<div style="font-family:sans-serif;max-width:700px"><svg viewBox="0 0 640 96" width="100%" role="img" aria-label="Progress map">{"".join(parts)}</svg>'
            f'<div style="font-size:14px">{line}</div><div style="font-size:12px;opacity:.6">{who}</div></div>')


_shown = []


def show_tracker():
    html = HTML(_html(_load()))
    if _shown:
        try:
            update_display(html, display_id="mml-tracker")
            return
        except Exception:
            pass
    display(html, display_id="mml-tracker")
    _shown.append(1)


def checkpoint(cid):
    """Call after an exercise passes: saves it, tells the instructor (if online), refreshes the map."""
    s = _load()
    _who(s)
    if cid not in s["done"]:
        s["done"][cid] = time.strftime("%H:%M:%S")
        s["queue"].append({"name": s["name"], "roll": s["roll"], "mission": MISSION, "checkpoint": cid, "ts": s["done"][cid], "sent": []})
    _save(s)
    sync()
    show_tracker()


def _start():
    s = _load()
    _who(s)
    if CLOUD_URL and _sending(s):  # Colab gives every notebook a fresh disk, so ask what you already finished elsewhere
        try:
            q = urllib.parse.urlencode({"roll": s["roll"], "token": TOKEN})
            for c in json.load(urllib.request.urlopen(CLOUD_URL + "?" + q, timeout=2)).get("done", []):
                s["done"].setdefault(c, "earlier")
        except Exception:
            pass
    _save(s)
    show_tracker()


_start()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_images, load_digits

np.random.seed(0)
photo = load_sample_images().images[1]                  # a bundled sample photo (flower)
img = photo.mean(axis=2) / 255.0                         # grayscale matrix with entries in [0, 1]
print("image matrix shape:", img.shape)
plt.imshow(img, cmap="gray"); plt.axis("off"); plt.show()

> **Use your own photo (optional).** Upload a picture to Colab (folder icon on the left), then run:
> ```python
> from PIL import Image
> img = np.asarray(Image.open("my_photo.jpg").convert("L").resize((400, 300))) / 255.0
> ```

## 1. A picture is a matrix; the SVD splits it into layers
$A=U\Sigma V^\top=\sum_i\sigma_i\,u_iv_i^\top$, a sum of rank-one "layers" ordered by importance $\sigma_1\ge\sigma_2\ge\dots$

In [ ]:
U, s, Vt = np.linalg.svd(img, full_matrices=False)
print("U", U.shape, " s", s.shape, " Vt", Vt.shape)
plt.semilogy(s); plt.xlabel("index i"); plt.ylabel("singular value (log scale)"); plt.title("how fast do the layers fade?"); plt.show()

## 2. Keep only the top $k$ layers (Core)
$A_k=\sum_{i\le k}\sigma_iu_iv_i^\top = U_k\,\mathrm{diag}(s_k)\,V_k^\top$. Fill in the reconstruction.

<details><summary>Hint</summary>

`U[:, :k]` is the first `k` columns, `s[:k]` the first `k` singular values, `Vt[:k, :]` the first `k` rows. Multiply columns by `s[:k]` with `U[:, :k] * s[:k]`.
</details>

In [ ]:
def compress(U, s, Vt, k):
    return ____            # TODO: (U[:, :k] * s[:k]) @ Vt[:k, :]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, k in zip(axes, [2, 10, 40, 150]):
    ax.imshow(compress(U, s, Vt, k), cmap="gray"); ax.axis("off"); ax.set_title(f"k = {k}")
plt.show()

## 3. How much did we save, and how much did we lose?
Storage: the original needs $mn$ numbers; $A_k$ needs only $k(m+n+1)$.

> **Math box (Eckart-Young).** $A_k$ is the *best* rank-$k$ approximation in Frobenius norm, and the error is exactly
> $$\|A-A_k\|_F=\sqrt{\sigma_{k+1}^2+\sigma_{k+2}^2+\cdots}.$$
> The code below checks this identity numerically.

In [ ]:
m, n = img.shape
for k in [5, 20, 50, 100]:
    err = np.linalg.norm(img - compress(U, s, Vt, k))          # Frobenius norm
    formula = np.sqrt((s[k:] ** 2).sum())
    ratio = k * (m + n + 1) / (m * n)
    print(f"k={k:<4} error={err:8.4f}  formula={formula:8.4f}  storage={ratio:6.1%}")
    assert np.isclose(err, formula)
print("Eckart-Young identity verified ✔")
checkpoint("m03-ey")

**Challenge:** find the smallest $k$ with storage under 20% where the picture still looks good to you. Post it in the group chat!

In [ ]:
ks = np.arange(1, 200)
errs = [np.sqrt((s[k:] ** 2).sum()) / np.linalg.norm(img) for k in ks]
plt.plot(ks, errs); plt.xlabel("k"); plt.ylabel("relative error"); plt.grid(alpha=.3); plt.show()

## 4. Same maths, new name: PCA (and "eigen-digits")
PCA finds the directions of maximum variance, which are the eigenvectors of the covariance matrix (*ML-II Unit IV*). Faces give "eigenfaces"; here we use handwritten digits, which is exactly the same computation.
**Core:** fill in the covariance matrix of the centred data.

In [ ]:
X = load_digits().data
mean = X.mean(axis=0)
Xc = X - mean                                   # centre the data
C = ____                                        # TODO: covariance matrix, (1/(n-1)) Xc^T Xc
vals, vecs = np.linalg.eigh(C)
order = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]
print("top 5 eigenvalues:", np.round(vals[:5], 1))

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
axes[0].imshow(mean.reshape(8, 8), cmap="gray_r"); axes[0].set_title("mean"); axes[0].axis("off")
for k in range(5):
    axes[k + 1].imshow(vecs[:, k].reshape(8, 8), cmap="coolwarm"); axes[k + 1].set_title(f"PC {k+1}"); axes[k + 1].axis("off")
plt.show()

explained = np.cumsum(vals) / vals.sum()
plt.plot(explained, "o-"); plt.axhline(0.9, color="r", ls="--"); plt.xlabel("components"); plt.ylabel("variance explained"); plt.show()
print("components for 90% of variance:", int(np.argmax(explained >= 0.9)) + 1, "out of 64")

# SVD and PCA agree: singular values of Xc relate to the eigenvalues of C
sv = np.linalg.svd(Xc, compute_uv=False)
assert np.allclose(sv ** 2 / (len(X) - 1), vals[:len(sv)].clip(min=0), atol=1e-6)
print("PCA eigenvalues = (singular values)^2 / (n-1) ✔")
checkpoint("m03-pca")

In [ ]:
checkpoint("m03-done")                      # Mission 3 complete: watch your map update at the top

## Stretch
Project every digit onto the first 2 principal components and scatter-plot, coloured by digit. Which digits separate, which overlap?

## Quest (take-home)
Run the compressor on your own photo. Report the smallest $k$ that keeps relative error under 5% and what fraction of storage that is. Bonus: compress each RGB channel separately.

## You just used...
- the **SVD** and **Eckart-Young theorem** (best low-rank approximation)
- **PCA** as an eigenproblem on the covariance matrix (ML-II Unit IV), and its link to the SVD

## Your CV line
> *Implemented image compression via truncated SVD and PCA in NumPy, verifying the Eckart-Young error formula and the SVD-PCA equivalence numerically.*

Next: **Mission 4**, rank the web and invent names with a Markov chain.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/04_mini_google.ipynb)